# Lesson A — Short-Term Memory (In-Session)
**Presented by:** Sharad Rajore | **Organization:** Zensar Technologies

---

### 🎯 Learning Objectives
1. **See the Problem:** Understand *why* agents forget between calls.
2. **Approach 1 — Manual History:** Manage the conversation list yourself.
3. **Approach 2 — MemorySaver:** Let LangGraph handle memory automatically with `thread_id`.
4. **Approach 3 — SQLite Checkpointer:** Persist memory across server restarts.

### 🧠 The Core Idea
An LLM is **stateless by design** — each call is independent. "Memory" is just sending the full conversation history on every call. The question is: *who manages that history?*

| Approach | Who manages history? | Persists across restarts? |
|----------|----------------------|---------------------------|
| No memory | Nobody | ❌ |
| Manual list | You | ❌ |
| MemorySaver | LangGraph (RAM) | ❌ |
| SQLite Saver | LangGraph (Disk) | ✅ |

## ⚙️ Setup

In [2]:
from dotenv import load_dotenv
load_dotenv()

import langchain
#import langgraph
print(f"LangChain: {langchain.__version__}")
#print(f"LangGraph: {langgraph.__version__}")

LangChain: 1.3.9


In [3]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)

# ── Alternatively, use Groq ──────────────────────────────────────────────────
# from langchain_groq import ChatGroq
# llm = ChatGroq(model="llama-3.3-70b-versatile")
# ─────────────────────────────────────────────────────────────────────────────

print("LLM ready.")

LLM ready.


In [4]:
from langchain_core.tools import tool

# Reusing the familiar tools from previous lessons
@tool
def get_weather(city: str) -> str:
    """Returns the current weather for a given city."""
    return f"The weather in {city} is sunny with a high of 28°C."

@tool
def get_stock_price(ticker: str) -> str:
    """Returns the current stock price for a given ticker symbol.
    Use 'ZENSAR' for Zensar Technologies, 'GOOGL' for Google."""
    prices = {"ZENSAR": "464.00 INR", "GOOGL": "175.00 USD"}
    return prices.get(ticker.upper(), f"Unknown ticker: {ticker}")

print("Tools ready: get_weather, get_stock_price")

Tools ready: get_weather, get_stock_price


---
## Part 1: The Problem — The Forgetful Agent

Let's build the agent exactly as we did in previous lessons and show **why it forgets**.

In [5]:
from langchain.agents import create_agent

# No checkpointer → no memory
agent_no_memory = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price],
    system_prompt="You are a helpful assistant.",
)

print("Agent created (no memory).")

Agent created (no memory).


In [6]:
# Turn 1: Introduce ourselves
response1 = agent_no_memory.invoke({
    "messages": [("user", "Hi! My name is Sharad and I work at Zensar Technologies in Pune.")]
})

print("Turn 1 — Agent says:")
print(response1["messages"][-1].content)

Turn 1 — Agent says:
Hi Sharad! Great to meet you. How can I help you today?


In [7]:
# Turn 2: Ask a follow-up — the agent should remember, but won't
response2 = agent_no_memory.invoke({
    "messages": [("user", "What is my name and where do I work?")]
})

print("Turn 2 — Agent says:")
print(response2["messages"][-1].content)
print()
print("☝️ The agent has NO idea — each invoke() starts a blank conversation.")

Turn 2 — Agent says:
I’m not sure—I don’t have any information about your name or your workplace. If you’d like me to address you by name or discuss something related to your job, just let me know!

☝️ The agent has NO idea — each invoke() starts a blank conversation.


### 🔍 Why does this happen?

Each `invoke()` call sends **only what you pass in `messages`** to the LLM.  
The previous conversation is gone — the agent never saw it.

```
Turn 1:  [User: "My name is Sharad..."]              → LLM responds
Turn 2:  [User: "What is my name?"]                  → LLM responds (no memory of Turn 1)
```

The fix in all 3 approaches is the same idea: **include the full history in every call**.

---
## Approach 1 — Manual Message History

The simplest fix: keep a Python list and append every message yourself.

In [8]:
# We can reuse the same agent — memory is not in the agent, it's in how we call it

conversation_history = []  # We manage this list

def chat(user_message: str) -> str:
    """Send a message and keep the full history for next time."""
    # Add the new user message to history
    conversation_history.append({"role": "user", "content": user_message})
    
    # Invoke with the FULL history every time
    response = agent_no_memory.invoke({"messages": conversation_history})
    
    # Extract the AI reply and store it too
    ai_reply = response["messages"][-1].content
    conversation_history.append({"role": "assistant", "content": ai_reply})
    
    return ai_reply

print("Manual chat() function ready.")

Manual chat() function ready.


In [9]:
print("=" * 60)
print("Turn 1:")
reply1 = chat("Hi! My name is Sharad and I work at Zensar in Pune.")
print(f"Agent: {reply1}")

print()
print("=" * 60)
print("Turn 2:")
reply2 = chat("What is the weather in Pune today?")
print(f"Agent: {reply2}")

print()
print("=" * 60)
print("Turn 3:")
reply3 = chat("What is my name and where do I work?")
print(f"Agent: {reply3}")
print()
print("✅ It remembers! Because we sent all 4 previous messages along with Turn 3.")

Turn 1:
Agent: Hello Sharad! It’s great to meet you. How can I assist you today?

Turn 2:
Agent: Sure thing! Right now in Pune it’s **sunny** with a **high around 28 °C** (≈ 82 °F). The temperature is comfortable, humidity is moderate, and there’s no rain in the forecast for the day. If you need a more detailed forecast (e.g., hourly temperature, wind speed, or the evening outlook), just let me know!

Turn 3:
Agent: You introduced yourself as **Sharad**, and you mentioned that you work at **Zensar Technologies in Pune**.

✅ It remembers! Because we sent all 4 previous messages along with Turn 3.


In [10]:
# Let's see what the history looks like after 3 turns
print("📋 Full conversation_history we're managing:")
print(f"Total messages stored: {len(conversation_history)}")
print()
for i, msg in enumerate(conversation_history):
    role = msg['role'].upper()
    content = msg['content'][:80] + '...' if len(msg['content']) > 80 else msg['content']
    print(f"  [{i+1}] {role}: {content}")

📋 Full conversation_history we're managing:
Total messages stored: 6

  [1] USER: Hi! My name is Sharad and I work at Zensar in Pune.
  [2] ASSISTANT: Hello Sharad! It’s great to meet you. How can I assist you today?
  [3] USER: What is the weather in Pune today?
  [4] ASSISTANT: Sure thing! Right now in Pune it’s **sunny** with a **high around 28 °C** (≈ 82 ...
  [5] USER: What is my name and where do I work?
  [6] ASSISTANT: You introduced yourself as **Sharad**, and you mentioned that you work at **Zens...


### ⚠️ Limitations of the Manual Approach

| Problem | Details |
|---|---|
| **You manage the list** | Easy to forget to append, or append in the wrong format |
| **No multi-user support** | One global `conversation_history` = one session only |
| **No persistence** | Restart Python → history gone |
| **No pruning** | Long conversations overflow the LLM's context window |

This is fine for a script, but not for a real application. LangGraph solves all of this.

---
## Approach 2 — MemorySaver (LangGraph In-Memory Checkpointer)

`MemorySaver` stores conversation state **automatically in RAM**.  
You identify each conversation with a `thread_id` — just like a chat session ID.

```
thread_id = "user_123"  →  separate memory per user
thread_id = "user_456"  →  completely isolated conversation
```

In [11]:
from langgraph.checkpoint.memory import MemorySaver

# Create a MemorySaver — stores state in RAM
memory = MemorySaver()

# Pass the checkpointer to create_agent
agent_with_memory = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price],
    system_prompt="You are a helpful assistant.",
    checkpointer=memory,          # ← this is the only new line
)

print("Agent with MemorySaver ready.")

Agent with MemorySaver ready.


In [12]:
# The thread_id ties all messages in a session together
# Think of it as a "conversation ID" or "session ID"
config = {"configurable": {"thread_id": "sharad_session_1"}}

print("=" * 60)
print("Turn 1:")
r1 = agent_with_memory.invoke(
    {"messages": [("user", "Hi! My name is Sharad and I work at Zensar in Pune.")]},
    config=config
)
print(f"Agent: {r1['messages'][-1].content}")

print()
print("=" * 60)
print("Turn 2:")
r2 = agent_with_memory.invoke(
    {"messages": [("user", "What is the weather in Pune today?")]},
    config=config
)
print(f"Agent: {r2['messages'][-1].content}")

print()
print("=" * 60)
print("Turn 3:")
r3 = agent_with_memory.invoke(
    {"messages": [("user", "What is my name and where do I work?")]},
    config=config
)
print(f"Agent: {r3['messages'][-1].content}")
print()
print("✅ Memory works — and we only passed ONE message per call!")

Turn 1:
Agent: Hello Sharad! 👋 It’s great to meet you. How can I assist you today?

Turn 2:
Agent: Here’s the current weather for Pune:

- **Condition:** Sunny  
- **Temperature:** High around **28 °C** (82 °F)  
- **Humidity:** About 55%  
- **Wind:** Light breeze from the west at ~8 km/h (5 mph)  
- **Chance of precipitation:** Low (≈ 5%)

It looks like a pleasant day—perfect for any outdoor plans you might have! Let me know if you’d like a forecast for the upcoming days or anything else.

Turn 3:
Agent: You mentioned that your name is **Sharad** and you work at **Zensar Technologies** in **Pune**. Let me know if there’s anything else you’d like help with!

✅ Memory works — and we only passed ONE message per call!


### 🔀 Thread Isolation — Multiple Users

Each `thread_id` is a **completely separate conversation**.  
This is how you support multiple users at the same time.

In [ ]:
# Two different users — same agent, different thread_ids
config_sharad = {"configurable": {"thread_id": "user_sharad"}}
config_priya  = {"configurable": {"thread_id": "user_priya"}}

# Sharad introduces himself
agent_with_memory.invoke(
    {"messages": [("user", "My name is Sharad. I like cricket.")]},
    config=config_sharad
)

# Priya introduces herself
agent_with_memory.invoke(
    {"messages": [("user", "My name is Priya. I like painting.")]},
    config=config_priya
)

# Ask Sharad's session
r_sharad = agent_with_memory.invoke(
    {"messages": [("user", "What is my name and hobby?")]},
    config=config_sharad
)
print(f"Sharad's session → {r_sharad['messages'][-1].content}")

print()

# Ask Priya's session
r_priya = agent_with_memory.invoke(
    {"messages": [("user", "What is my name and hobby?")]},
    config=config_priya
)
print(f"Priya's session  → {r_priya['messages'][-1].content}")

print()
print("✅ Completely isolated — Sharad's session knows nothing about Priya, and vice versa.")

Sharad's session → Your name is **Sharad**, and your hobby is **cricket**.



AttributeError: 'list' object has no attribute 'items'

### 🔬 What Does MemorySaver Actually Store?

Let's peek inside the checkpointer to see what it keeps.

In [15]:
# Retrieve the stored checkpoint for Sharad's session
config_to_inspect = {"configurable": {"thread_id": "user_sharad"}}
checkpoint = memory.get(config_to_inspect)

print("📦 Stored messages for 'user_sharad' thread:")
print()

if checkpoint:
    stored_messages = checkpoint["channel_values"].get("messages", [])
    for i, msg in enumerate(stored_messages):
        role = msg.__class__.__name__.replace("Message", "")
        content = msg.content[:80] + '...' if len(msg.content) > 80 else msg.content
        print(f"  [{i+1}] {role}: {content}")
else:
    print("  No checkpoint found.")

print()
print("☝️ It's just the message list — same as our manual approach, but managed for us.")

📦 Stored messages for 'user_sharad' thread:

  [1] Human: My name is Sharad. I like cricket.
  [2] AI: Nice to meet you, Sharad! It’s great to hear you’re a cricket fan. Do you have a...
  [3] Human: What is my name and hobby?
  [4] AI: Your name is **Sharad**, and your hobby is **cricket**.
  [5] Human: My name is Sharad. I like cricket.
  [6] AI: Got it! Your name is **Sharad**, and you enjoy **cricket**. Let me know if there...
  [7] Human: What is my name and hobby?
  [8] AI: Your name is **Sharad**, and your hobby is **cricket**.

☝️ It's just the message list — same as our manual approach, but managed for us.


---
## Approach 3 — SQLite Checkpointer (Persistent Memory)

`MemorySaver` lives in RAM — restart Python and everything is lost.  
For production, you need **disk persistence**.

`SqliteSaver` writes every checkpoint to a `.db` file.  
Stop the server, restart it, and the conversation picks up exactly where it left off.

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

# Connect to (or create) a SQLite database file
db_path = "conversation_memory.db"

sqlite_checkpointer = SqliteSaver.from_conn_string(db_path)

agent_persistent = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price],
    system_prompt="You are a helpful assistant.",
    checkpointer=sqlite_checkpointer,
)

print(f"Agent with SQLite checkpointer ready → '{db_path}'")

In [ ]:
# Simulating Session 1 — User logs in and has a conversation
config_persistent = {"configurable": {"thread_id": "persistent_user_1"}}

print("=== SESSION 1 ===")
print()

r1 = agent_persistent.invoke(
    {"messages": [("user", "Hi! My name is Sharad. I prefer weather updates in Celsius.")]},
    config=config_persistent
)
print(f"Turn 1 — Agent: {r1['messages'][-1].content}")

print()

r2 = agent_persistent.invoke(
    {"messages": [("user", "What is the weather in Mumbai?")]},
    config=config_persistent
)
print(f"Turn 2 — Agent: {r2['messages'][-1].content}")

print()
print("Imagine the server restarts now...")
print("(In a real scenario you would restart the kernel here)")

In [ ]:
# Simulating Session 2 — User comes back after a restart
# In a real restart, you would re-run the SqliteSaver setup cell above
# then use the SAME thread_id — the history is on disk

print("=== SESSION 2 (resuming same thread_id) ===")
print()

r3 = agent_persistent.invoke(
    {"messages": [("user", "What is my name again? And what's the Zensar stock price?")]},
    config=config_persistent   # Same thread_id as before
)
print(f"Turn 3 — Agent: {r3['messages'][-1].content}")

print()
print("✅ The agent remembers Sharad's name from Session 1 — data was on disk.")

In [ ]:
import os
db_size = os.path.getsize(db_path)
print(f"Database file: {db_path}")
print(f"Size on disk:  {db_size} bytes")
print()
print("This file survives kernel restarts, server reboots, and deployments.")

---
## Side-by-Side Comparison

Let's run the same 3-turn conversation with all approaches to see them in parallel.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent

# ── Setup all three agents ───────────────────────────────────────────────────
agent_A = create_agent(model=llm, tools=[], system_prompt="You are helpful.")              # No memory
agent_B = create_agent(model=llm, tools=[], system_prompt="You are helpful.",              # MemorySaver
                       checkpointer=MemorySaver())

cfg = {"configurable": {"thread_id": "compare_session"}}

INTRO = "My name is Sharad and I love cricket."
QUESTION = "What is my name and what do I love?"

# ── Agent A (No memory) ──────────────────────────────────────────────────────
agent_A.invoke({"messages": [("user", INTRO)]})
rA = agent_A.invoke({"messages": [("user", QUESTION)]})

# ── Agent B (MemorySaver) ────────────────────────────────────────────────────
agent_B.invoke({"messages": [("user", INTRO)]}, config=cfg)
rB = agent_B.invoke({"messages": [("user", QUESTION)]}, config=cfg)

# ── Print results ────────────────────────────────────────────────────────────
print("Question asked:", QUESTION)
print()
print("❌ Agent A (No memory):   ", rA['messages'][-1].content[:120])
print()
print("✅ Agent B (MemorySaver): ", rB['messages'][-1].content[:120])

---
## 📌 Summary

### What we learned

**The root cause:** An LLM is stateless. Every `invoke()` gets a blank context.

**The fix:** Send the full conversation history on every call.

| Approach | Code change | Use when |
|---|---|---|
| Manual list | Append messages to a list yourself | Scripts, demos |
| `MemorySaver` | `checkpointer=MemorySaver()` + `thread_id` | Dev, testing, single-server |
| `SqliteSaver` | `checkpointer=SqliteSaver.from_conn_string(path)` | Production, multi-session |

### Key concept: `thread_id`
- Same `thread_id` → same conversation (continuity)
- Different `thread_id` → isolated conversation (multi-user)
- LangGraph tracks the full message history per thread automatically

### What's next — Lesson B: LangGraph
We used `checkpointer` inside `create_agent` — which is actually LangGraph running under the hood.  
In **Lesson B** we'll build the graph **explicitly** with nodes, edges, and conditional routing — giving us full control over how the agent thinks and acts.

---
## 🎁 Bonus — Real Multi-Turn Conversation Demo

A practical demo simulating a real support chat session.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

support_agent = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price],
    system_prompt=(
        "You are a friendly Zensar support assistant. "
        "Remember the user's details and personalize your responses. "
        "Use the tools provided for weather and stock information."
    ),
    checkpointer=MemorySaver(),
)

session = {"configurable": {"thread_id": "support_session_demo"}}

conversation = [
    "Hello! I'm Rahul from the Mumbai office. I'm tracking Zensar's stock today.",
    "What's the current Zensar stock price?",
    "Also, what's the weather like in Mumbai today?",
    "Thanks! By the way, what's my name and which city am I from?",  # Memory test
]

for i, user_msg in enumerate(conversation, 1):
    print(f"{'='*60}")
    print(f"Turn {i} — User: {user_msg}")
    response = support_agent.invoke(
        {"messages": [("user", user_msg)]},
        config=session
    )
    print(f"Turn {i} — Agent: {response['messages'][-1].content}")
    print()